In [1]:
import pandas as pd
import requests
import json
import os
import boto3
import re
from cities import cities
from dotenv import load_dotenv

# Weather data

In [2]:
# Get the latitude and longitude of the cities
cities_location = []

headers = {
    "User-Agent": "QHA"
}

for city in cities:
    r_location = requests.get(f"https://nominatim.openstreetmap.org/search?format=json&city={city}", headers=headers)
    location_data = r_location.json()

    current_city = {
        "name": location_data[0]['name'],
        "lat": location_data[0]['lat'],
        "lon": location_data[0]['lon'],
    }

    cities_location.append(current_city)
print(cities_location)



[{'name': 'Mont-Saint-Michel', 'lat': '46.7798558', 'lon': '-75.3362610'}, {'name': 'St. Malo', 'lat': '49.3146950', 'lon': '-96.9538228'}, {'name': 'Bayeux', 'lat': '49.2764624', 'lon': '-0.7024738'}, {'name': 'Le Havre', 'lat': '49.4938975', 'lon': '0.1079732'}, {'name': 'Rouen', 'lat': '49.4404591', 'lon': '1.0939658'}, {'name': 'Paris', 'lat': '48.8588897', 'lon': '2.3200410'}, {'name': 'Amiens', 'lat': '49.8941708', 'lon': '2.2956951'}, {'name': 'Lille', 'lat': '50.6365654', 'lon': '3.0635282'}, {'name': 'Strasbourg', 'lat': '48.5846140', 'lon': '7.7507127'}, {'name': 'Château du Haut-Kœnigsbourg', 'lat': '48.2495226', 'lon': '7.3454923'}, {'name': 'Colmar', 'lat': '48.0777517', 'lon': '7.3579641'}, {'name': 'Eguisheim', 'lat': '48.0447968', 'lon': '7.3079618'}, {'name': 'Besançon', 'lat': '47.2380222', 'lon': '6.0243622'}, {'name': 'Dijon', 'lat': '47.3215806', 'lon': '5.0414701'}, {'name': 'Annecy', 'lat': '45.8992348', 'lon': '6.1288847'}, {'name': 'Grenoble', 'lat': '45.187560

In [3]:
df_location = pd.DataFrame(cities_location)
df_location

,name,lat,lon
0,Mont-Saint-Michel,46.7798558,-75.3362610
1,St. Malo,49.3146950,-96.9538228
2,Bayeux,49.2764624,-0.7024738
3,Le Havre,49.4938975,0.1079732
4,Rouen,49.4404591,1.0939658
5,Paris,48.8588897,2.3200410
6,Amiens,49.8941708,2.2956951
7,Lille,50.6365654,3.0635282
8,Strasbourg,48.5846140,7.7507127
9,Château du Haut-Kœnigsbourg,48.2495226,7.3454923


In [4]:
load_dotenv()
api_weather_key = os.getenv("OPENWEATHER_API_KEY")

params = {
    "appid": api_weather_key,
    "units": "metric",
    "lang": "fr",
    "cnt": 5,
}

lat = 44.8333
lon = -0.5667
cnt = 16

url_test = f"https://api.openweathermap.org/data/2.5/forecast?lat=44.8333&lon=-0.5667&units=metric&lang=fr&cnt=7&appid={api_weather_key}"

r = requests.get(url_test)
data = r.json()

data['list']



[{'dt': 1747753200,
  'main': {'temp': 22.75,
   'feels_like': 22.52,
   'temp_min': 22.26,
   'temp_max': 22.75,
   'pressure': 1020,
   'sea_level': 1020,
   'grnd_level': 1015,
   'humidity': 55,
   'temp_kf': 0.49},
  'weather': [{'id': 802,
    'main': 'Clouds',
    'description': 'partiellement nuageux',
    'icon': '03d'}],
  'clouds': {'all': 29},
  'wind': {'speed': 5.29, 'deg': 309, 'gust': 6.22},
  'visibility': 10000,
  'pop': 0,
  'sys': {'pod': 'd'},
  'dt_txt': '2025-05-20 15:00:00'},
 {'dt': 1747764000,
  'main': {'temp': 20.02,
   'feels_like': 19.51,
   'temp_min': 18.54,
   'temp_max': 20.02,
   'pressure': 1021,
   'sea_level': 1021,
   'grnd_level': 1016,
   'humidity': 55,
   'temp_kf': 1.48},
  'weather': [{'id': 802,
    'main': 'Clouds',
    'description': 'partiellement nuageux',
    'icon': '03d'}],
  'clouds': {'all': 39},
  'wind': {'speed': 4.67, 'deg': 300, 'gust': 6.47},
  'visibility': 10000,
  'pop': 0,
  'sys': {'pod': 'd'},
  'dt_txt': '2025-05-20 18

In [5]:
all_weather = []
weather_params = {
    "appid": api_weather_key,
    "units": "metric",
    "lang": "fr",
    "cnt": 7,
}

for index,weather in enumerate(cities_location):
    temp_score = 0
    rain = 0
    wind = 0
    wind_score = 0
    rain_score = 0
    
    r_weather = requests.get(f"https://api.openweathermap.org/data/2.5/forecast?lat={weather['lat']}&lon={weather['lon']}", params=weather_params)
    current_weather = r_weather.json()['list']

    for i in range(len(current_weather)):
        temp_score = round(temp_score + (current_weather[i]['main']['feels_like'] / len(current_weather)), 2)
        rain = current_weather[i].get('rain', 0)
        
        if rain != 0:
            rain_key = list(current_weather[i]['rain'].keys())[0]
            match = re.search(r'\d+', rain_key)
            hours = 0

            if match:
                hours = int(match.group())

            rain_daily_score = hours * current_weather[i]['rain'][rain_key]
            rain_score = round(rain_score + (rain_daily_score / len(current_weather)), 2)
        else:
            rain_score = rain_score + 0

        wind = round(wind + (current_weather[i]['wind']['speed'] / len(current_weather)), 2)

        beaufort_score = 0 # Beaufort scale
        if current_weather[i]['wind']['speed'] == 0:
            beaufort_score = 0
        elif current_weather[i]['wind']['speed'] < 5:
            beaufort_score = 1
        elif current_weather[i]['wind']['speed'] < 11:
            beaufort_score = 2
        elif current_weather[i]['wind']['speed'] < 19:
            beaufort_score = 3
        elif current_weather[i]['wind']['speed'] < 28:
            beaufort_score = 4
        elif current_weather[i]['wind']['speed'] < 38:
            beaufort_score = 5
        elif current_weather[i]['wind']['speed'] < 49:
            beaufort_score = 6
        elif current_weather[i]['wind']['speed'] < 61:
            beaufort_score = 7
        elif current_weather[i]['wind']['speed'] < 74:
            beaufort_score = 8
        elif current_weather[i]['wind']['speed'] < 88:
            beaufort_score = 9
        elif current_weather[i]['wind']['speed'] < 102:
            beaufort_score = 10
        elif current_weather[i]['wind']['speed'] < 117:
            beaufort_score = 11
        elif current_weather[i]['wind']['speed'] > 117:
            beaufort_score = 12

        if current_weather[i]['main']['feels_like'] < 25 or current_weather[i]['wind']['speed'] > 30:
            wind_penalty = beaufort_score
        else:
            wind_penalty = 0

        wind_score = round(wind_score + (wind_penalty / len(current_weather)), 2)

    weather_score = round(temp_score - rain_score - wind_score, 3)


    current_weather_data = {
        "index" : index,
        "name" : weather['name'],
        "temperature_mean" : temp_score,
        "rain_mean" : rain_score,
        "wind_score" : wind_score,
        "score" : weather_score,
    }
    all_weather.append(current_weather_data)
       
all_weather


[{'index': 0,
  'name': 'Mont-Saint-Michel',
  'temperature_mean': 3.01,
  'rain_mean': 0.13,
  'wind_score': 0.98,
  'score': 1.9},
 {'index': 1,
  'name': 'St. Malo',
  'temperature_mean': 6.35,
  'rain_mean': 1.05,
  'wind_score': 1.73,
  'score': 3.57},
 {'index': 2,
  'name': 'Bayeux',
  'temperature_mean': 14.01,
  'rain_mean': 0,
  'wind_score': 1.13,
  'score': 12.88},
 {'index': 3,
  'name': 'Le Havre',
  'temperature_mean': 13.51,
  'rain_mean': 0.13,
  'wind_score': 1.28,
  'score': 12.1},
 {'index': 4,
  'name': 'Rouen',
  'temperature_mean': 16.25,
  'rain_mean': 0,
  'wind_score': 1.13,
  'score': 15.12},
 {'index': 5,
  'name': 'Paris',
  'temperature_mean': 17.11,
  'rain_mean': 0,
  'wind_score': 1.13,
  'score': 15.98},
 {'index': 6,
  'name': 'Amiens',
  'temperature_mean': 13.86,
  'rain_mean': 0,
  'wind_score': 1.13,
  'score': 12.73},
 {'index': 7,
  'name': 'Lille',
  'temperature_mean': 13.97,
  'rain_mean': 0,
  'wind_score': 1.13,
  'score': 12.84},
 {'index'

In [6]:
df_weather = pd.DataFrame(all_weather)
df_weather = df_weather.sort_values(by='score', ascending=False)
df_weather

,index,name,temperature_mean,rain_mean,wind_score,score
22,22,Avignon,18.82,0.00,1.28,17.54
27,27,Collioure,19.41,0.00,1.87,17.54
25,25,Aigues-Mortes,19.41,0.00,1.88,17.53
24,24,Nîmes,18.17,0.00,0.98,17.19
20,20,Marseille,19.25,0.11,2.03,17.11
23,23,Uzès,17.87,0.00,0.98,16.89
8,8,Strasbourg,17.95,0.12,0.98,16.85
26,26,Saintes-Maries-de-la-Mer,18.26,0.00,1.88,16.38
19,19,Cassis,18.18,0.15,2.03,16.00
21,21,Aix-en-Provence,17.85,0.27,1.58,16.00


In [7]:
df_weather.to_csv("export/weather.csv", index=False)

# Hotel data

In [8]:
!python hotel.py

2025-05-20 14:50:54 [scrapy.utils.log] INFO: Scrapy 2.11.1 started (bot: scrapybot)
2025-05-20 14:50:54 [scrapy.utils.log] INFO: Versions: lxml 5.2.1.0, libxml2 2.13.1, cssselect 1.2.0, parsel 1.8.1, w3lib 2.1.2, Twisted 23.10.0, Python 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 08:28:27) [Clang 14.0.6 ], pyOpenSSL 24.2.1 (OpenSSL 3.0.15 3 Sep 2024), cryptography 43.0.0, Platform macOS-10.16-x86_64-i386-64bit
2025-05-20 14:50:54 [scrapy.addons] INFO: Enabled addons:
[]
2025-05-20 14:50:54 [py.warnings] WARNING: /opt/anaconda3/lib/python3.12/site-packages/scrapy/utils/request.py:254: ScrapyDeprecationWarning: '2.6' is a deprecated value for the 'REQUEST_FINGERPRINTER_IMPLEMENTATION' setting.

It is also the default value. In other words, it is normal to get this warning if you have not defined a value for the 'REQUEST_FINGERPRINTER_IMPLEMENTATION' setting. This is so for backward compatibility reasons, but it will change in a future version of Scrapy.

See the documentati

# Merge data in one csv

In [9]:
with open("export/hotels.json", "r") as f:
    hotel_data = json.load(f)

df_hotel = pd.DataFrame(hotel_data)
df_hotel = df_hotel.rename(columns={""
    "name": "hotel_name",
    "link": "hotel_link",
    "latitude": "lat",
    "longitude": "lon",
    "description": "desc",
})
df_hotel[df_hotel["city_id"]== 0]

,city_id,city,hotel_name,hotel_link,lat,lon,desc,stars,rating
0,0,Mont Saint Michel,Gîte La Mouette de 4 personnes,https://www.booking.com/hotel/fr/gite-la-mouet...,48.615795851658,-1.488698244809,Hébergement géré par un particulier,0,"9,3"
1,0,Mont Saint Michel,Maison au pied du Mont Saint Michel 2,https://www.booking.com/hotel/fr/maison-au-pie...,48.6156117,-1.4885121,"Offrant une vue sur le jardin, l’hébergement M...",0,"9,3"
2,0,Mont Saint Michel,Auberge de la Baie,https://www.booking.com/hotel/fr/auberge-de-la...,48.61599317655586,-1.4882531762123108,"Situé dans la campagne normande, cet hôtel dis...",2,"8,3"
290,0,Mont Saint Michel,Mon Saint Michel,https://www.booking.com/hotel/fr/mon-saint-mic...,48.613627,-1.485791,"Offrant une vue sur le jardin, l’établissement...",0,"8,8"
291,0,Mont Saint Michel,Gites Bellevue,https://www.booking.com/hotel/fr/gites-bellevu...,48.60788006624136,-1.5172237157821655,Hébergement géré par un particulier,0,"9,3"
292,0,Mont Saint Michel,gites 2 beauvoir,https://www.booking.com/hotel/fr/gites-2-beauv...,48.5976913,-1.5047171,Hébergement géré par un particulier,0,"8,7"
294,0,Mont Saint Michel,A l ombre du Mont Saint Michel,https://www.booking.com/hotel/fr/a-l-ombre-du-...,48.6156258,-1.4651207,"Situé à Huisnes-sur-Mer, l’établissement A l o...",0,"9,5"
295,0,Mont Saint Michel,Le Marquis De La Guintre,https://www.booking.com/hotel/fr/le-marquis-de...,48.6248647057706,-1.44534185528755,"Situé à Courtils, l’établissement Le Marquis D...",0,"8,8"
296,0,Mont Saint Michel,Apparthôtel Mont Saint Michel - Résidence Fleu...,https://www.booking.com/hotel/fr/residence-fle...,48.596482169806976,-1.5031469166023044,L’Apparthôtel Mont Saint Michel - Résidence Fl...,3,"8,3"
297,0,Mont Saint Michel,Vent des Grèves,https://www.booking.com/hotel/fr/vent-des-grev...,48.615403,-1.49144,"Offrant une vue sur le jardin, l’établissement...",0,"9,1"


In [10]:
df_weather = df_weather.rename(columns={
    "index": "city_id",
    "name": "city_name",
})
df_weather[df_weather["city_id"]== 0]


,city_id,city_name,temperature_mean,rain_mean,wind_score,score
0,0,Mont-Saint-Michel,3.01,0.13,0.98,1.9


In [33]:
import numpy as np

df_global = pd.merge(df_weather, df_hotel, on='city_id', how='inner')
df_global = df_global.drop(columns=["city"])
df_global = df_global.rename(columns={"city_name": "city"})
df_global["stars"] = df_global["stars"].replace(0, np.nan)
df_global.sample(10)


,city_id,city,temperature_mean,rain_mean,wind_score,score,hotel_name,hotel_link,lat,lon,desc,stars,rating
153,8,Strasbourg,17.95,0.12,0.98,16.85,Hôtel Tandem - Boutique Hôtel,https://www.booking.com/hotel/fr/ha-tel-nid-de...,48.58383632696426,7.734948992729187,Ce boutique hôtel respectueux de l’environneme...,4.0,"9,0"
670,13,Dijon,16.60,2.95,0.98,12.67,Cosy T2 38m2 - Centre-Ville Dijon- Gare/Darcy,https://www.booking.com/hotel/fr/9-rue-benigne...,47.3251439,5.0298452,Hébergement géré par un particulier,NaN,"9,3"
15,22,Avignon,18.82,0.00,1.28,17.54,Hotel De Cambis Best Western Premier Collection,https://www.booking.com/hotel/fr/de-cambis-bw-...,43.947113,4.80333,L’établissement Hotel De Cambis Best Western P...,4.0,"8,7"
410,11,Eguisheim,18.16,2.70,0.98,14.48,Eguisheim village préféré des français grand s...,https://www.booking.com/hotel/fr/studio-eguish...,48.0476831,7.31329149999999,Hébergement géré par un particulier,NaN,"9,2"
664,13,Dijon,16.60,2.95,0.98,12.67,Kyriad Hotel Dijon Gare,https://www.booking.com/hotel/fr/kyriad-gare.f...,47.32279603784554,5.029898285865784,"Situé à 120 mètres de la gare de Dijon, le Kyr...",3.0,"7,1"
370,18,Bormes-les-Mimosas,16.34,0.48,0.98,14.88,Bormes - Le Loft,https://www.booking.com/hotel/fr/bormes-le-lof...,43.1507533,6.3406583,Hébergement géré par un particulier,NaN,"9,1"
340,10,Colmar,18.45,2.46,0.98,15.01,Hotel Arc-En-Ciel Colmar Contact Hotel,https://www.booking.com/hotel/fr/arc-en-ciel-h...,48.08745399108503,7.360686957836151,L'Hotel Arc-En-Ciel Colmar Contact Hotel vous ...,2.0,"8,3"
866,0,Mont-Saint-Michel,3.01,0.13,0.98,1.90,Le Saint Aubert,https://www.booking.com/hotel/fr/hotel-saint-a...,48.612937834706464,-1.5101051330566406,"Niché dans un écrin de verdure, à seulement 2 ...",3.0,"7,7"
363,18,Bormes-les-Mimosas,16.34,0.48,0.98,14.88,Boulevard du Soleil,https://www.booking.com/hotel/fr/boulevard-du-...,43.149232813225,6.341970146281,Hébergement géré par un particulier,NaN,"9,4"
806,17,Rougon,11.70,1.56,0.98,9.16,Camping Notre Dame,https://www.booking.com/hotel/fr/camping-notre...,43.84570690185817,6.50452584028244,Ce terrain de camping à la gestion familiale e...,3.0,"8,9"


In [38]:
df_global.to_csv("export/weather_and_hotels.csv", index=False)

# Data to S3

In [41]:
import boto3

In [42]:
session = boto3.Session()

In [43]:
s3 = session.resource("s3")

In [ ]:
s3.Bucket("qha-kayak").upload_file("export/weather_and_hotels.csv", "weather_and_hotels.csv")